<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Hierarchical Agent Teams
</b></font> </br></p>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M21-Hierarchical-Agent-Teams"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 35.231.110.37
Hostname: 37.110.231.35.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/New_

# 1 | Übersicht
---

<p><font color='black' size="5">
Flat vs. Hierarchisch
</font></p>

In *Multi-Agent Patterns*/*Supervisor Pattern* haben wir **flache Multi-Agent-Systeme** kennengelernt: ein Supervisor
koordiniert N Worker-Agenten direkt. Bei wachsender Komplexität entstehen drei typische Failure-Modes:

| Failure Mode | Problem |
|---|---|
| **Context Dilution** | Das ursprüngliche Ziel geht im Rauschen der Zwischenschritte verloren — das Modell verliert den roten Faden |
| **Tool Saturation** | Je mehr Tools, desto schlechter die Selektion — falsche Tools, ungültige Argumente |
| **Lost in the Middle** | LLMs untergewichten Inhalte in der Mitte langer Kontextfenster — auch klare Anweisungen werden übersehen |

**Hierarchische Agent Teams** lösen das durch gezielte Delegation:

| Merkmal | Flat (M19/M20) | Hierarchisch (M24) |
|---------|---------------|-------------------|
| **Ebenen** | 2 (Supervisor + Worker) | 3+ (Supervisor → Lead → Worker) |
| **Supervisor-Aufgabe** | Alle Worker direkt steuern | Nur Teams delegieren |
| **Skalierung** | Schwer (viele Worker = komplexerer Supervisor) | Leicht (neue Teams hinzufügen) |
| **Spezialisierung** | Supervisor kennt alle Details | Leads kennen ihre Domäne |
| **Typischer Einsatz** | 2–5 Agenten | 6+ Agenten, klare Domänen-Trennung |

# 2 | Hierarchische Multi-Agent-Systeme
---

<p><font color='black' size="5">
Drei-Ebenen-Architektur
</font></p>

Eine hierarchische Architektur teilt Komplexität in klare Verantwortungsebenen:

| Ebene | Rolle | Modell | Aufgabe |
|-------|-------|--------|---------|
| **Ebene 1** | Top-Level Supervisor | `o3` | Strategische Entscheidungen, Team-Routing |
| **Ebene 2** | Team Leads | `o3-mini` | Taktische Koordination innerhalb des Teams |
| **Ebene 3** | Specialist Workers | `gpt-4o-mini` | Operative Ausführung, Tool-Nutzung |

**Delegation statt Kontrolle:** Jede Ebene kennt nur die Ebene direkt unter sich.
Der Top Supervisor weiß nicht, wie der Research Lead intern seine Spezialisten koordiniert.

**Beispiel: Forschungsbericht erstellen**
```
User: 'Erstelle einen Bericht über LangGraph'
  └─ Top Supervisor: 'Erst Research, dann Writing'
       ├─ Research Lead: 'Searcher, dann Analyst'
       │    ├─ Web Searcher: web_suche('LangGraph') → Rohdaten
       │    └─ Data Analyst: analysiere(Rohdaten) → strukturierte Fakten
       └─ Writing Lead: 'Writer, dann Editor'
            ├─ Content Writer: schreibe(Fakten) → Entwurf
            └─ Editor: editiere(Entwurf) → finaler Bericht
```

---

**Wie die Hierarchie die drei Failure-Modes löst:**

**Context Dilution → Contextual Packets**
Statt der vollständigen Gesprächshistorie bekommt jeder Agent nur einen relevanten Kontext-Ausschnitt.
Der Top Supervisor hält den globalen State — an den JSON-Formatter unten schickt er aber kein 4.000-Wort-Strategiedokument.
Das Signal-Rausch-Verhältnis bleibt hoch auf jeder Ebene.

**Tool Saturation → Principle of Least Privilege**
Jeder Agent erhält nur die Tools, die er für seine Aufgabe braucht.
Der Security-Agent bekommt den Vulnerability-Scanner — nicht das CI/CD-Pipeline-Tool.
Das gehört dem DevOps-Agent. Schmale, zweckgebundene Toolboxen statt einer großen gemeinsamen Auswahl.

**Lost in the Middle → Model Flexibility**
Nicht jede Aufgabe braucht das teuerste Frontier-Modell.
Komplexe Planung und Decomposition oben: `o3`.
Modulare, selbstständige Aufgaben unten: `gpt-4o-mini`.
Jede Ebene bekommt genau das Modell, das ihre Aufgabe erfordert.

In [14]:
#@markdown   <p><font size="4" color='green'>  Drei-Ebenen-Hierarchie</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    SUP(["🔵 Top Supervisor\no3\nStrategische Steuerung"])

    subgraph rt ["Research Team"]
        RL["🟣 Research Lead\no3-mini"]
        W1["⚪ Web Searcher\ngpt-4o-mini"]
        W2["⚪ Data Analyst\ngpt-4o-mini"]
        RL --> W1
        RL --> W2
    end

    subgraph wt ["Writing Team"]
        WL["🟣 Writing Lead\no3-mini"]
        W3["⚪ Content Writer\ngpt-4o-mini"]
        W4["⚪ Editor\ngpt-4o-mini"]
        WL --> W3
        WL --> W4
    end

    SUP -->|delegiert| RL
    SUP -->|delegiert| WL

    style SUP fill:#1565C0,color:#fff
    style RL  fill:#4A148C,color:#fff
    style WL  fill:#4A148C,color:#fff
    style W1  fill:#37474F,color:#fff
    style W2  fill:#37474F,color:#fff
    style W3  fill:#37474F,color:#fff
    style W4  fill:#37474F,color:#fff
'''
mermaid(diagram, width=900)

<p><font color='black' size="5">
Tool-Delegation: Agents als Tools
</font></p>

Das eleganteste Muster für Hierarchien: **Agents als Tools**.
Jeder Lead ist ein ReAct-Agent, der seine Spezialisten-Agenten als Tools aufruft.
Der Top Supervisor ruft die Team-Leads ebenfalls als Tools auf.

```python
# Ebene 3: Worker als ReAct-Agent
searcher_agent = create_react_agent(worker_llm, tools=[web_suche])

# Ebene 2: Lead ruft Worker via @tool auf
@tool
def call_searcher(query: str) -> str:
    result = searcher_agent.invoke({'messages': [HumanMessage(query)]})
    return result['messages'][-1].content

research_lead = create_react_agent(lead_llm, tools=[call_searcher, call_analyst])

# Ebene 1: Supervisor ruft Team via @tool auf
@tool
def call_research_team(aufgabe: str) -> str:
    result = research_lead.invoke({'messages': [HumanMessage(aufgabe)]})
    return result['messages'][-1].content

top_supervisor = create_react_agent(supervisor_llm, tools=[call_research_team, call_writing_team])
```

**Vorteil:** Keine manuelle StateGraph-Verwaltung pro Team. Jede Ebene verhält sich
autonom. Neue Teams lassen sich ohne Änderung der bestehenden Hierarchie ergänzen.

In [3]:
#@markdown   <p><font size="4" color='green'>  ⚙️ LLM-Setup (Modell-Auswahl-Guide)</font> </br></p>

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

# Modell-Auswahl-Guide: Regeln 1, 1b, 4
supervisor_llm = init_chat_model("openai:o3")                            # Ebene 1: Top Supervisor (kritisch)
lead_llm       = init_chat_model("openai:o3-mini")                       # Ebene 2: Team Leads (einfaches Routing)
worker_llm     = init_chat_model("openai:gpt-4o-mini", temperature=0.0)  # Ebene 3: Specialists

zeilen = [
    "## ⚙️ Modell-Zuweisung (Modell_Auswahl_Guide v1.2)", "",
    "| Ebene | Rolle | Modell | temperature |",
    "|-------|-------|--------|-------------|",
    "| Ebene 1 | Top Supervisor (kritisch) | `o3` | ❌ nicht setzen |",
    "| Ebene 2 | Team Leads (einfaches Routing) | `o3-mini` | ❌ nicht setzen |",
    "| Ebene 3 | Specialist Workers | `gpt-4o-mini` | ✅ `0.0` ok |",
]
mprint("\n".join(zeilen))

## ⚙️ Modell-Zuweisung (Modell_Auswahl_Guide v1.2)

| Ebene | Rolle | Modell | temperature |
|-------|-------|--------|-------------|
| Ebene 1 | Top Supervisor (kritisch) | `o3` | ❌ nicht setzen |
| Ebene 2 | Team Leads (einfaches Routing) | `o3-mini` | ❌ nicht setzen |
| Ebene 3 | Specialist Workers | `gpt-4o-mini` | ✅ `0.0` ok |

# 3 | Team-Lead → Specialists
---

<p><font color='black' size="5">
Aufbau: Research Team + Writing Team
</font></p>

Das System wird **Bottom-Up** aufgebaut:

1. **Ebene 3:** Domain-Tools + Worker-Agenten (ReAct mit Tool-Nutzung)
2. **Ebene 2:** Team-Lead-Agenten (rufen Workers via `@tool` auf)
3. **Ebene 1:** Top-Supervisor (ruft Teams via `@tool` auf)

Jede Ebene ist ein vollständig autonomer ReAct-Agent mit klar definierter Rolle.

In [15]:
#@markdown   <p><font size="4" color='green'>  Tool-Delegation durch alle Ebenen</font> </br></p>

diagram = '''
sequenceDiagram
    actor U as Nutzer
    participant S  as Top Supervisor (o3)
    participant RL as Research Lead (o3-mini)
    participant WS as Web Searcher (gpt-4o-mini)
    participant DA as Data Analyst (gpt-4o-mini)
    participant WL as Writing Lead (o3-mini)
    participant CW as Content Writer (gpt-4o-mini)
    participant ED as Editor (gpt-4o-mini)

    U->>S: Aufgabe: Bericht erstellen

    S->>RL: call_research_team(aufgabe)
    activate RL
    RL->>WS: call_searcher(query)
    WS-->>RL: Rohdaten
    RL->>DA: call_analyst(rohdaten)
    DA-->>RL: Strukturierte Fakten
    RL-->>S: Research-Ergebnis
    deactivate RL

    S->>WL: call_writing_team(aufgabe + fakten)
    activate WL
    WL->>CW: call_writer(fakten)
    CW-->>WL: Entwurf
    WL->>ED: call_editor(entwurf)
    ED-->>WL: Finaler Text
    WL-->>S: Writing-Ergebnis
    deactivate WL

    S-->>U: Fertige Antwort
'''
mermaid(diagram, width=1200)

In [5]:
#@markdown   <p><font size="4" color='green'>  🔍 Ebene 3 + 2: Research Team aufbauen</font> </br></p>

# ── Ebene 3: Domain-Tools ────────────────────────────────────────────────────
@tool
def web_suche(query: str) -> str:
    '''Sucht nach aktuellen Informationen zu einem Thema im Web.'''
    wissen = {
        "langchain": "LangChain: Framework fuer LLM-Anwendungen (Chains, Agents, Tools, LCEL).",
        "langgraph": "LangGraph: Zustandsbasierte Multi-Agent-Graphen auf Basis von LangChain.",
        "langsmith": "LangSmith: Tracing, Evaluation und Monitoring fuer LLM-Pipelines.",
        "openai":    "OpenAI: Anbieter von GPT-4o, gpt-5.1, o3-Modellen und Embeddings.",
        "rag":       "RAG (Retrieval-Augmented Generation): Kombination von Vektorsuche und LLM.",
    }
    key = query.lower().split()[0].rstrip('?.:,')
    treffer = wissen.get(key, f'Allgemeine KI-Informationen zu "{query}" gefunden.')
    return f'[Web-Suche] {treffer}'

@tool
def daten_analyse(rohtext: str) -> str:
    '''Analysiert und strukturiert Rohdaten aus der Recherche.'''
    punkte = [p.strip() for p in rohtext.split('.') if len(p.strip()) > 15]
    kernaussagen = ' | '.join(punkte[:3]) if punkte else rohtext[:120]
    return f'[Analyse] {len(punkte)} Kernaussagen: {kernaussagen}'

# ── Ebene 3: Worker-Agenten ───────────────────────────────────────────────────
searcher_agent = create_react_agent(
    worker_llm, tools=[web_suche],
    prompt="Du bist Web-Recherche-Spezialist. Nutze immer web_suche. Antworte auf Deutsch."
)
analyst_agent = create_react_agent(
    worker_llm, tools=[daten_analyse],
    prompt="Du bist Daten-Analyst. Nutze immer daten_analyse. Antworte auf Deutsch."
)

# ── Ebene 2: Research-Lead-Tools ──────────────────────────────────────────────
@tool
def call_searcher(query: str) -> str:
    '''Beauftragt den Web-Searcher-Spezialisten mit einer Suchanfrage.'''
    result = searcher_agent.invoke(
        {"messages": [HumanMessage(query)]},
        config={"recursion_limit": 10}
    )
    return result["messages"][-1].content

@tool
def call_analyst(rohdaten: str) -> str:
    '''Beauftragt den Daten-Analysten mit der Auswertung von Rohdaten.'''
    result = analyst_agent.invoke(
        {"messages": [HumanMessage(rohdaten)]},
        config={"recursion_limit": 10}
    )
    return result["messages"][-1].content

# ── Ebene 2: Research Lead Agent ──────────────────────────────────────────────
RESEARCH_LEAD_PROMPT = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m21_research_lead_prompt.md",
    mode="S",
)
research_lead = create_react_agent(
    lead_llm,
    tools=[call_searcher, call_analyst],
    prompt=RESEARCH_LEAD_PROMPT,
)
mprint("✅ Research Team: Web Searcher + Data Analyst → Research Lead")

✅ Research Team: Web Searcher + Data Analyst → Research Lead

In [6]:
#@markdown   <p><font size="4" color='green'>  ✍️ Ebene 3 + 2: Writing Team aufbauen</font> </br></p>

# ── Ebene 3: Writing-Tools ───────────────────────────────────────────────────
@tool
def text_schreiben(thema: str, stichpunkte: str) -> str:
    '''Schreibt einen strukturierten Fliesstext zu einem Thema.'''
    return (
        f'[Content Writer] Entwurf zu "{thema}":\n\n'
        f'{stichpunkte}\n\n'
        f'Kernpunkte wurden in lesbare Prosa ueberfuehrt.'
    )

@tool
def text_editieren(text: str) -> str:
    '''Ueberarbeitet und poliert einen bestehenden Textentwurf.'''
    bereinigt = ' '.join(text.split())
    return (
        f'[Editor] Ueberarbeiteter Text:\n\n{bereinigt}\n\n'
        f'Lesbarkeit und Struktur verbessert.'
    )

# ── Ebene 3: Worker-Agenten ───────────────────────────────────────────────────
writer_agent = create_react_agent(
    worker_llm, tools=[text_schreiben],
    prompt="Du bist Content Writer. Nutze text_schreiben fuer alle Schreibaufgaben. Antworte auf Deutsch."
)
editor_agent = create_react_agent(
    worker_llm, tools=[text_editieren],
    prompt="Du bist Editor. Nutze text_editieren zur Ueberarbeitung von Texten. Antworte auf Deutsch."
)

# ── Ebene 2: Writing-Lead-Tools ───────────────────────────────────────────────
@tool
def call_writer(aufgabe: str) -> str:
    '''Beauftragt den Content Writer mit einer Schreibaufgabe.'''
    result = writer_agent.invoke(
        {"messages": [HumanMessage(aufgabe)]},
        config={"recursion_limit": 10}
    )
    return result["messages"][-1].content

@tool
def call_editor(text: str) -> str:
    '''Beauftragt den Editor mit der Ueberarbeitung eines Textes.'''
    result = editor_agent.invoke(
        {"messages": [HumanMessage(text)]},
        config={"recursion_limit": 10}
    )
    return result["messages"][-1].content

# ── Ebene 2: Writing Lead Agent ───────────────────────────────────────────────
WRITING_LEAD_PROMPT = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m21_writing_lead_prompt.md",
    mode="S",
)
writing_lead = create_react_agent(
    lead_llm,
    tools=[call_writer, call_editor],
    prompt=WRITING_LEAD_PROMPT,
)
mprint("✅ Writing Team: Content Writer + Editor → Writing Lead")

✅ Writing Team: Content Writer + Editor → Writing Lead

In [7]:
#@markdown   <p><font size="4" color='green'>  🏗️ Ebene 1: Top-Level Supervisor</font> </br></p>

# ── Ebene 1: Team-Tools ──────────────────────────────────────────────────────
@tool
def call_research_team(aufgabe: str) -> str:
    '''Delegiert eine Recherche-Aufgabe an das Research-Team (Lead + Searcher + Analyst).'''
    result = research_lead.invoke(
        {"messages": [HumanMessage(aufgabe)]},
        config={"recursion_limit": 15}
    )
    return result["messages"][-1].content

@tool
def call_writing_team(aufgabe: str) -> str:
    '''Delegiert eine Schreibaufgabe an das Writing-Team (Lead + Writer + Editor).'''
    result = writing_lead.invoke(
        {"messages": [HumanMessage(aufgabe)]},
        config={"recursion_limit": 15}
    )
    return result["messages"][-1].content

# ── Ebene 1: Top Supervisor ───────────────────────────────────────────────────
SUPERVISOR_PROMPT = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m21_supervisor_prompt.md",
    mode="S",
)
top_supervisor = create_react_agent(
    supervisor_llm,
    tools=[call_research_team, call_writing_team],
    prompt=SUPERVISOR_PROMPT,
)

mprint(
    "## 🏗️ Hierarchie aufgebaut\n\n"
    "```\n"
    "Ebene 1: Top Supervisor (o3)\n"
    "  ├─ call_research_team → Research Lead (o3-mini)\n"
    "  │    ├─ call_searcher → Web Searcher (gpt-4o-mini)\n"
    "  │    └─ call_analyst  → Data Analyst (gpt-4o-mini)\n"
    "  └─ call_writing_team  → Writing Lead (o3-mini)\n"
    "       ├─ call_writer   → Content Writer (gpt-4o-mini)\n"
    "       └─ call_editor   → Editor (gpt-4o-mini)\n"
    "```"
)

## 🏗️ Hierarchie aufgebaut

```
Ebene 1: Top Supervisor (o3)
  ├─ call_research_team → Research Lead (o3-mini)
  │    ├─ call_searcher → Web Searcher (gpt-4o-mini)
  │    └─ call_analyst  → Data Analyst (gpt-4o-mini)
  └─ call_writing_team  → Writing Lead (o3-mini)
       ├─ call_writer   → Content Writer (gpt-4o-mini)
       └─ call_editor   → Editor (gpt-4o-mini)
```

In [8]:
#@markdown   <p><font size="4" color='green'>  🚀 Hierarchisches Team starten</font> </br></p>

import time

aufgabe = "Erklaere LangGraph und schreibe eine kurze, verstaendliche Zusammenfassung auf Deutsch."

print(f"Aufgabe: {aufgabe}")
print()

start = time.perf_counter()
result = top_supervisor.invoke(
    {"messages": [HumanMessage(aufgabe)]},
    config={"recursion_limit": 25, "run_name": "m21-Kap3-hierarchie-demo", "tags": ["m21"]}
)
latenz = round((time.perf_counter() - start) * 1000)

finale_antwort = result["messages"][-1].content

zeilen = [
    "## 🏗️ Ausführungsverlauf — Message History", "",
    "| # | Typ | Inhalt (gekürzt) |",
    "|---|-----|------------------|",
]
for i, msg in enumerate(result["messages"], 1):
    typ = type(msg).__name__
    inhalt = getattr(msg, 'content', '') or ''
    if isinstance(inhalt, list):
        inhalt = str(inhalt[0])[:80] if inhalt else ''
    zeilen.append(f"| {i} | `{typ}` | {str(inhalt)[:90]} |")
zeilen += ["", f"**Latenz gesamt:** `{latenz} ms`"]
mprint("\n".join(zeilen))

print()
mprint(f"## 💬 Finale Antwort\n\n{finale_antwort}")

Aufgabe: Erklaere LangGraph und schreibe eine kurze, verstaendliche Zusammenfassung auf Deutsch.



## 🏗️ Ausführungsverlauf — Message History

| # | Typ | Inhalt (gekürzt) |
|---|-----|------------------|
| 1 | `HumanMessage` | Erklaere LangGraph und schreibe eine kurze, verstaendliche Zusammenfassung auf Deutsch. |
| 2 | `AIMessage` |  |
| 3 | `ToolMessage` | Hier eine kurze, verständliche Zusammenfassung zu LangGraph:

LangGraph ist ein innovative |
| 4 | `AIMessage` |  |
| 5 | `ToolMessage` | LangGraph ist eine umfassende Plattform zur Erstellung und Verwaltung von Sprachmodellen,  |
| 6 | `AIMessage` | LangGraph ist ein Open-Source-Framework, mit dem sich Arbeitsabläufe rund um große Sprachm |

**Latenz gesamt:** `108195 ms`

## 💬 Finale Antwort

LangGraph ist ein Open-Source-Framework, mit dem sich Arbeitsabläufe rund um große Sprachmodelle (LLMs) als übersichtlicher Graph bauen lassen. Jede Aufgabe – etwa ein API-Aufruf, eine Datenprüfung oder ein Modell-Prompt – wird als Knoten dargestellt; die Pfeile zeigen, in welcher Reihenfolge oder unter welchen Bedingungen die Schritte ablaufen.  

Hauptziele  
• Komplexe LLM-Prozesse sichtbar machen und verständlich halten  
• Änderungen erleichtern: Knoten lassen sich hinzufügen, entfernen oder austauschen, ohne den ganzen Ablauf neu zu schreiben  
• Besseres Debugging und Logging, weil jeder Zwischenschritt separat einsehbar ist  

Wichtige Funktionen  
• Visueller Editor zum Zeichnen und Testen der Graphen  
• Enge Anbindung an LangChain – vorhandene Chains können direkt als Knoten genutzt werden  
• Ereignis- und zustandsbasierte Verzweigungen für nicht-lineare Abläufe  
• Erweiterbarkeit: eigene Python- oder JavaScript-Funktionen, externe APIs und Datenquellen lassen sich einbinden  

Typische Einsatzgebiete  
– Chatbots oder Agenten, die mehrere Tools kombinieren (Recherche, Datenbank, Berechnung)  
– Automatisierte Content-Pipelines, z. B. Zusammenfassen, Übersetzen und Publizieren von Texten  
– Entscheidungsunterstützungssysteme, bei denen Ergebnisse mehrerer Modelle zusammengeführt werden  

Kurz gesagt: LangGraph bietet eine grafische, modulare Schicht über LangChain & Co. Das spart Zeit beim Entwickeln, erleichtert Wartung und macht selbst komplexe KI-Workflows transparent und nachvollziehbar.

# 4 | Parallele vs. sequenzielle Ausführung
---

<p><font color='black' size="5">
Wann parallel, wann sequenziell?
</font></p>

Die Ausführungsreihenfolge hat großen Einfluss auf Latenz und Ergebnisqualität:

| Kriterium | Sequenziell | Parallel |
|-----------|-------------|----------|
| **Datenabhängigkeit** | Team B braucht Ergebnis von Team A | Teams sind unabhängig |
| **Latenz** | Addiert sich | Entspricht dem langsamsten Team |
| **Kontext-Sharing** | B kennt A's Ergebnis | Kein automatisches Sharing |
| **Typischer Einsatz** | Research → Writing | Parallel-Analyse mehrerer Domänen |
| **Python API** | Synchron `invoke()` | `asyncio.gather()` |

**Sequenziell:** Writing baut auf Research-Fakten auf — klare Abhängigkeit.
**Parallel:** Zwei Research-Teams durchsuchen verschiedene Quellen gleichzeitig — unabhängig.

In [9]:
#@markdown   <p><font size="4" color='green'>  Sequenziell vs. Parallel</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    subgraph seq ["Sequenziell — abhängige Teams"]
        direction LR
        SS["Supervisor"]
        SR["Research\nTeam"]
        SW["Writing\nTeam"]
        SS -->|1. delegiert| SR
        SR -->|2. Fakten| SS
        SS -->|3. delegiert\n+ Fakten| SW
        SW -->|4. Text| SS
    end

    subgraph par ["Parallel — unabhängige Teams"]
        direction LR
        PS["Supervisor"]
        PA["Research\nTeam A"]
        PB["Research\nTeam B"]
        PS -->|gleichzeitig| PA
        PS -->|gleichzeitig| PB
        PA & PB -->|join| PS
    end

    style SS fill:#1565C0,color:#fff
    style SR fill:#4A148C,color:#fff
    style SW fill:#4A148C,color:#fff
    style PS fill:#1565C0,color:#fff
    style PA fill:#2E7D32,color:#fff
    style PB fill:#2E7D32,color:#fff
'''
mermaid(diagram, width=900)

In [10]:
#@markdown   <p><font size="4" color='green'>  📏 Sequenzielle Ausführung: Research → Writing</font> </br></p>

import time

THEMA = "LangSmith"

def sequenziell_ausfuehren(thema: str) -> dict:
    '''Research → Writing: Writing bekommt Research-Ergebnis als Kontext.'''
    start = time.perf_counter()

    # Phase 1: Research Team
    t1 = time.perf_counter()
    research_result = research_lead.invoke(
        {"messages": [HumanMessage(f"Recherchiere alles ueber: {thema}")]},
        config={"recursion_limit": 10}
    )
    research_ms  = round((time.perf_counter() - t1) * 1000)
    research_text = research_result["messages"][-1].content

    # Phase 2: Writing Team (kennt Research-Ergebnis)
    t2 = time.perf_counter()
    writing_result = writing_lead.invoke(
        {"messages": [HumanMessage(
            f"Schreibe eine Zusammenfassung zu {thema}.\n"
            f"Recherche-Grundlage: {research_text[:300]}"
        )]},
        config={"recursion_limit": 10}
    )
    writing_ms = round((time.perf_counter() - t2) * 1000)
    total_ms   = round((time.perf_counter() - start) * 1000)

    return {
        "research_ms": research_ms,
        "writing_ms":  writing_ms,
        "total_ms":    total_ms,
        "modus":       "sequenziell",
    }

seq = sequenziell_ausfuehren(THEMA)
mprint(
    f"✅ **Sequenziell:** Research `{seq['research_ms']} ms`"
    f" → Writing `{seq['writing_ms']} ms`"
    f" = **`{seq['total_ms']} ms`** gesamt"
)

✅ **Sequenziell:** Research `20816 ms` → Writing `20571 ms` = **`41388 ms`** gesamt

In [11]:
#@markdown   <p><font size="4" color='green'>  ⚡ Parallele Ausführung: ThreadPoolExecutor</font> </br></p>

from concurrent.futures import ThreadPoolExecutor

# Zwei unabhängige Research-Aufgaben für parallele Ausführung
THEMA_A = "LangChain"
THEMA_B = "LangGraph"

def run_research(thema: str) -> str:
    '''Führt Research für ein Thema aus.'''
    result = research_lead.invoke(
        {"messages": [HumanMessage(f"Recherchiere: {thema}")]},
        config={"recursion_limit": 10}
    )
    return result["messages"][-1].content

def parallel_ausfuehren(thema_a: str, thema_b: str) -> dict:
    '''Beide Research-Tasks gleichzeitig starten – ThreadPoolExecutor für sync invoke().'''
    start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=2) as pool:
        future_a = pool.submit(run_research, thema_a)
        future_b = pool.submit(run_research, thema_b)
        ergebnis_a = future_a.result()
        ergebnis_b = future_b.result()

    total_ms = round((time.perf_counter() - start) * 1000)
    return {
        "total_ms":  total_ms,
        "modus":     "parallel",
        "thema_a":   thema_a,
        "thema_b":   thema_b,
    }

par = parallel_ausfuehren(THEMA_A, THEMA_B)
mprint(
    f"⚡ **Parallel:** {par['thema_a']} + {par['thema_b']} gleichzeitig"
    f" = **`{par['total_ms']} ms`** gesamt"
)

⚡ **Parallel:** LangChain + LangGraph gleichzeitig = **`16396 ms`** gesamt

In [12]:
#@markdown   <p><font size="4" color='green'>  📊 Vergleich: Sequenziell vs. Parallel</font> </br></p>

speedup = seq["total_ms"] / par["total_ms"] if par["total_ms"] > 0 else 1.0

zeilen = [
    "## ⚡ Ausführungs-Vergleich", "",
    "| Dimension | Sequenziell | Parallel |",
    "|-----------|-------------|----------|",
    f"| **Gesamtdauer** | `{seq['total_ms']} ms` | `{par['total_ms']} ms` |",
    f"| **Speedup** | 1.0× (Baseline) | `{speedup:.1f}×` schneller |",
    "| **Kontext-Sharing** | Writing kennt Research-Ergebnis | Kein automatisches Sharing |",
    "| **Ideal wenn** | Team B braucht Ergebnis von Team A | Teams arbeiten unabhängig |",
    "| **Python API** | `research_lead.invoke()` sequenziell | `ThreadPoolExecutor` |",
    "",
    "> **Faustregel:**",
    "> - ✅ **Sequenziell:** Writing braucht Research-Fakten als Input",
    "> - ✅ **Parallel:** Teams analysieren unabhängige Themen gleichzeitig",
    "> - ⚠️ Parallel spart Zeit, verhindert aber automatisches Kontext-Sharing",
    "> - 💡 `ThreadPoolExecutor` ist die richtige Wahl für synchrone `invoke()`-Aufrufe",
]
mprint("\n".join(zeilen))

## ⚡ Ausführungs-Vergleich

| Dimension | Sequenziell | Parallel |
|-----------|-------------|----------|
| **Gesamtdauer** | `41388 ms` | `16396 ms` |
| **Speedup** | 1.0× (Baseline) | `2.5×` schneller |
| **Kontext-Sharing** | Writing kennt Research-Ergebnis | Kein automatisches Sharing |
| **Ideal wenn** | Team B braucht Ergebnis von Team A | Teams arbeiten unabhängig |
| **Python API** | `research_lead.invoke()` sequenziell | `ThreadPoolExecutor` |

> **Faustregel:**
> - ✅ **Sequenziell:** Writing braucht Research-Fakten als Input
> - ✅ **Parallel:** Teams analysieren unabhängige Themen gleichzeitig
> - ⚠️ Parallel spart Zeit, verhindert aber automatisches Kontext-Sharing
> - 💡 `ThreadPoolExecutor` ist die richtige Wahl für synchrone `invoke()`-Aufrufe

In [13]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M21-Hierarchical-Agent-Teams", limit=3, show_steps=True)

## LangSmith Trace — `M21-Hierarchical-Agent-Teams`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `LangGraph` | ✅ success | 8.2s | 0 |
| `LangGraph` | ❌ pending | — | 0 |
| `LangGraph` | ✅ success | 20.6s | 0 |


### Steps — letzter Run: `LangGraph`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `chain` | `agent` | ✅ | 3.1s |
| 2 | `chain` | `tools` | ✅ | 3.1s |
| ↳ | `tool` | `call_searcher` | ✅ | 3.1s |
| 3 | `chain` | `agent` | ✅ | 1.9s |

# 5 | Grenzen & Risiken
---

<p><font color='black' size="5">
Wo hierarchische Systeme scheitern können
</font></p>

Hierarchische Agent Teams lösen reale Probleme — bringen aber eigene Risiken mit:

| Risiko | Beschreibung |
|---|---|
| **Task Decomposition** | Das gesamte System hängt an der Fähigkeit des High-Level-Agents, komplexe Ziele korrekt zu zerlegen. Fehlerhafte Decomposition (falsche Reihenfolge, fehlende Schritte, Over-Decomposition) pflanzt sich durch alle Ebenen fort. |
| **Orchestration Overhead** | State Management, Handoff-Logik zwischen Ebenen, Retry-Loops bei Fehlern — kein einzelner Agent mehr, sondern ein vollständiges System mit eigenen Fehlerquellen. |
| **Telephone Game Effect** | Leicht fehlerhafte Decomposition + Kontextpruning auf dem Weg nach unten → der spezialisierte Agent führt die falsche Aufgabe präzise aus. |
| **Recursive Error Loops** | Brittle Handoff-Logik → Agents geben Fehler durch die Hierarchie weiter, bis das Token-Limit erreicht ist. |

**Task Decomposition** ist das kritischste Risiko: LLMs planen inkonsistent.
Sie übersehen Abhängigkeiten, unterschätzen Komplexität — oder zerlegen einfache Aufgaben in unnötige Teilschritte.

**Recursive Feedback als Gegenmittel:**
Wenn ein Low-Level-Agent fertig ist, meldet er zurück. Der Supervisor kann das Ergebnis
als Quality Gate nutzen — bei Fehler Retry auslösen oder den Plan anpassen.
Das setzt voraus, dass die Handoff-Logik stabil ist und nicht selbst zur Fehlerquelle wird.

> Die Hierarchie verhindert, dass sich der Agent im Kontext verliert.
> Sie kann ihn aber im Organigramm verlieren lassen — wenn der Plan oben schon falsch war.

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Hierarchisches Agenten-Team erweitern
</font></p>

**Aufgabe 1 — Drittes Team: Fact-Checker:**
Ergänze das System um ein drittes Team `fact_checker_team`
mit einem Lead (`o3-mini`) und zwei Specialists (`gpt-4o-mini`):
- `call_verifier`: Verifiziert einzelne Fakten
- `call_sourcer`: Prüft Quellenangaben

Der Top Supervisor soll nach Research und Writing das Fact-Checker-Team aufrufen.

**Aufgabe 2 — Parallele Research-Teams:**
Erstelle zwei unabhängige Research-Teams (z.B. `tech_research` und `business_research`).
Starte beide **parallel** mit `asyncio.gather()` und übergebe die kombinierten Ergebnisse
anschließend sequenziell an das Writing Team.
Miss den Latenz-Unterschied gegenüber rein sequenzieller Ausführung.

**Aufgabe 3 — Qualitäts-Gate zwischen Teams:**
Implementiere einen `quality_check`-Tool für den Top Supervisor.
Nach dem Writing Team bewertet ein `o3`-Judge (Muster aus *LangSmith Evaluations Basics*) das Ergebnis
auf einer Skala 0.0–1.0. Bei Score < 0.7 sendet der Supervisor das Writing Team
erneut mit gezieltem Verbesserungs-Feedback (max. 2 Iterationen).

> 💡 **Tipp:** Die Tool-Delegation macht es einfach, neue Teams hinzuzufügen,
> ohne die bestehende Hierarchie zu ändern. Nutze `call_research_team` und
> `call_writing_team` als Vorlagen für neue Team-Tools.